# LAB 6 — Dashboard Comparativo com LangFuse
## Aula 4: BM25 vs Dense (BGE-M3) vs Hybrid RRF vs Neural Sparse (SPLADE) vs Hybrid + Contextual
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

---

**Pré-requisitos:** LAB1–LAB5 concluídos (índices populados, configs gravadas).

**Objetivo:**

1. Rodar as **5 estratégias** sobre as 20 queries do `queries_avaliacao_aula4.json` (corpus TCU 2026):
   - **BM25** puro (lexical)
   - **Dense kNN** (BGE-M3 via Ollama connector)
   - **Hybrid RRF** (BM25 + Dense, RRF k=60)
   - **Neural Sparse** (SPLADE multilíngue, LAB4)
   - **Hybrid + Contextual** (chunks enriquecidos, LAB5)
2. Medir **latência, MRR@10, Recall@10, NDCG@10** por estratégia
3. Registrar **scores** no **LangFuse** (Scores API) com `metadata` rico
4. Gerar **dashboard comparativo** (matplotlib + tabelas)
5. Quantificar a **melhoria do RAG** com cada técnica (Δ vs BM25)

## 1. Setup

In [ ]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'langfuse>=2.36.0', 'matplotlib', 'pandas',
            'seaborn', 'python-dotenv']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os, json, time, warnings
from opensearchpy import OpenSearch
from langfuse import Langfuse
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns
from dotenv import load_dotenv
warnings.filterwarnings('ignore'); load_dotenv()

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')
client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, ssl_show_warn=False,
)
print(f"OpenSearch {client.info()['version']['number']} OK")

In [ ]:
with open('lab1_config.json') as f: cfg1 = json.load(f)
with open('lab4_config.json') as f: cfg4 = json.load(f)
with open('lab5_config.json') as f: cfg5 = json.load(f)

INDEX_DENSE    = cfg1['index_name']
DENSE_MODEL_ID = cfg1['model_id']
INDEX_SPARSE   = cfg4['sparse_index']
SPARSE_MODEL_ID = cfg4['sparse_model_id']
INDEX_BASELINE   = cfg5['index_baseline']
INDEX_CONTEXTUAL = cfg5['index_contextual']
PIPELINE_RRF = 'hybrid_rrf_pipeline'

print('Índices:')
for n, ix in [('dense (LAB1)', INDEX_DENSE),
              ('sparse (LAB4)', INDEX_SPARSE),
              ('baseline chunks (LAB5)', INDEX_BASELINE),
              ('contextual chunks (LAB5)', INDEX_CONTEXTUAL)]:
    ok = client.indices.exists(index=ix)
    cnt = client.count(index=ix)['count'] if ok else 0
    print(f'  {n}: {ix} ({cnt} docs)')

## 2. Queries de Avaliação + LangFuse

In [ ]:
with open('../datasets/queries_avaliacao_aula4.json') as f:
    queries = json.load(f)
print(f'Queries: {len(queries)}')

LANGFUSE_HOST       = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')
try:
    lf = Langfuse(public_key=LANGFUSE_PUBLIC_KEY, secret_key=LANGFUSE_SECRET_KEY, host=LANGFUSE_HOST)
    trace = lf.trace(
        name='Aula4-LAB6-Dashboard-RAG-Improvement',
        metadata={
            'aula': '4', 'lab': '6',
            'corpus': 'TCU 2026 (1.100 acórdãos)',
            'n_queries': len(queries),
            'estrategias': ['BM25', 'Dense_BGE_M3', 'Hybrid_RRF',
                            'Neural_Sparse', 'Hybrid_Contextual']
        }
    )
    LANGFUSE_OK = True
    print(f'LangFuse OK — trace {trace.id}')
except Exception as e:
    LANGFUSE_OK = False; trace = None
    print(f'LangFuse indisponível: {e}')

## 3. Funções de Busca por Estratégia

Cada função retorna `(doc_ids, latência_ms)`. Para os índices da Aula 4 (LAB1 e LAB4) os IDs são o `id` do documento; para o índice de chunks (LAB5) usamos `doc_id` (origem do chunk).

In [ ]:
K = 10

def _ids(hits, field='id'):
    return [h['_source'].get(field, h['_id']) for h in hits]

def bm25(q, k=K):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_DENSE,
                      body={'size': k, 'query': {'match': {'conteudo': {'query': q}}},
                            '_source': ['id']})
    return _ids(r['hits']['hits']), (time.perf_counter()-t0)*1000

def dense(q, k=K):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_DENSE,
                      body={'size': k,
                            'query': {'neural': {'knn_vector': {
                                'query_text': q, 'model_id': DENSE_MODEL_ID, 'k': k}}},
                            '_source': ['id']})
    return _ids(r['hits']['hits']), (time.perf_counter()-t0)*1000

def hybrid_rrf(q, k=K):
    t0 = time.perf_counter()
    body = {'size': k,
            'query': {'hybrid': {'queries': [
                {'match':  {'conteudo': {'query': q}}},
                {'neural': {'knn_vector': {'query_text': q, 'model_id': DENSE_MODEL_ID, 'k': k}}},
            ]}},
            '_source': ['id']}
    r = client.search(index=INDEX_DENSE, body=body, params={'search_pipeline': PIPELINE_RRF})
    return _ids(r['hits']['hits']), (time.perf_counter()-t0)*1000

def neural_sparse(q, k=K):
    t0 = time.perf_counter()
    body = {'size': k,
            'query': {'neural_sparse': {'sparse_embedding': {
                'query_text': q, 'model_id': SPARSE_MODEL_ID}}},
            '_source': ['id']}
    r = client.search(index=INDEX_SPARSE, body=body)
    return _ids(r['hits']['hits']), (time.perf_counter()-t0)*1000

def hybrid_contextual(q, k=K):
    """Hybrid RRF sobre o índice CONTEXTUAL (chunks enriquecidos do LAB5).
    Como o índice contém chunks, devolvemos o doc_id (origem) deduplicado.
    """
    t0 = time.perf_counter()
    body = {'size': k * 3,  # mais resultados pq há vários chunks por doc
            'query': {'hybrid': {'queries': [
                {'match':  {'conteudo': {'query': q}}},
                {'neural': {'knn_vector': {'query_text': q, 'model_id': DENSE_MODEL_ID, 'k': k*3}}},
            ]}},
            '_source': ['doc_id']}
    r = client.search(index=INDEX_CONTEXTUAL, body=body, params={'search_pipeline': PIPELINE_RRF})
    seen, out = set(), []
    for h in r['hits']['hits']:
        d = h['_source']['doc_id']
        if d not in seen:
            seen.add(d); out.append(d)
        if len(out) == k:
            break
    return out, (time.perf_counter()-t0)*1000

STRATEGIES = {
    'BM25':              bm25,
    'Dense_BGE_M3':      dense,
    'Hybrid_RRF':        hybrid_rrf,
    'Neural_Sparse':     neural_sparse,
    'Hybrid_Contextual': hybrid_contextual,
}
print('Estratégias:', list(STRATEGIES))

## 4. Métricas e Benchmark

In [ ]:
def mrr(r, rel, k=K):
    for i, d in enumerate(r[:k], 1):
        if d in rel: return 1.0 / i
    return 0.0
def recall(r, rel, k=K):
    return (len(set(r[:k]) & set(rel)) / len(rel)) if rel else 0.0
def ndcg(r, rel, k=K):
    dcg  = sum((1.0 if d in rel else 0.0) / np.log2(i+1) for i, d in enumerate(r[:k], 1))
    idcg = sum(1.0 / np.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg/idcg if idcg > 0 else 0.0

N_REPS = 3
rows = []
for q in queries:
    rel = q['documentos_relevantes']
    for nome, fn in STRATEGIES.items():
        lats, last = [], []
        for _ in range(N_REPS):
            try:
                ids, lat = fn(q['texto'])
                lats.append(lat); last = ids
            except Exception as e:
                print(f'{nome} fail: {e}')
                lats.append(9999.0); last = []
        lat_med = float(np.mean(lats))
        m  = mrr(last, rel);  r  = recall(last, rel);  n  = ndcg(last, rel)
        rows.append({'query_id': q['id'], 'estrategia': nome,
                     'MRR@10': m, 'Recall@10': r, 'NDCG@10': n,
                     'latencia_ms': lat_med})
        if LANGFUSE_OK:
            sp = trace.span(name=f"{q['id']}-{nome}",
                            metadata={'query': q['texto'], 'k': K, 'reps': N_REPS,
                                       'top_ids': last})
            sp.score(name='MRR_10',     value=m)
            sp.score(name='Recall_10',  value=r)
            sp.score(name='NDCG_10',    value=n)
            sp.score(name='latency_ms', value=lat_med)
            sp.end()

df = pd.DataFrame(rows)
if LANGFUSE_OK: lf.flush()
print(f'Benchmark: {len(df)} registros')

## 5. Resumo + Δ vs BM25 (melhoria do RAG)

Para cada métrica calculamos a melhoria absoluta e relativa em relação à BM25 (referência clássica) — essa é a **principal evidência quantitativa de que as técnicas da Aula 4 melhoram o RAG**.

In [ ]:
agg = df.groupby('estrategia')[['MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']].mean().round(4)

base = agg.loc['BM25']
delta = pd.DataFrame({
    'Δ MRR@10':       (agg['MRR@10']    - base['MRR@10']).round(4),
    'Δ Recall@10':    (agg['Recall@10'] - base['Recall@10']).round(4),
    'Δ NDCG@10':      (agg['NDCG@10']   - base['NDCG@10']).round(4),
    '%↑MRR vs BM25':  ((agg['MRR@10']    / base['MRR@10']    - 1) * 100).round(1),
    '%↑Rec vs BM25':  ((agg['Recall@10'] / base['Recall@10'] - 1) * 100).round(1),
    '%↑NDCG vs BM25': ((agg['NDCG@10']   / base['NDCG@10']   - 1) * 100).round(1),
})
agg_full = agg.join(delta).sort_values('MRR@10', ascending=False)
print('===== BENCHMARK AULA 4 — Melhoria do RAG =====')
print(agg_full)

# Registrar agregados e deltas no LangFuse
if LANGFUSE_OK:
    for nome, row in agg_full.iterrows():
        for col, val in row.items():
            try:
                trace.score(name=f'{col}__{nome}'.replace(' ', '_'),
                            value=float(val),
                            comment=f'{col} para {nome}')
            except Exception:
                pass
    lf.flush()
    print(f'\nScores agregados em {LANGFUSE_HOST}/traces/{trace.id}')

## 6. RAGAS (Context Precision/Recall) — Resgate do LAB5

Adicionamos os scores RAGAS para visualizar lado a lado a melhoria de **retrieval (MRR/Recall/NDCG)** e a **qualidade de contexto (RAGAS)** atingida com Contextual Retrieval.

In [ ]:
ragas_df = pd.DataFrame([
    {'estrategia': 'Baseline (sem contexto)',
     'Context Precision': cfg5['cp_baseline'],
     'Context Recall':    cfg5['cr_baseline']},
    {'estrategia': 'Contextual Retrieval (#T09)',
     'Context Precision': cfg5['cp_contextual'],
     'Context Recall':    cfg5['cr_contextual']}
])
print('RAGAS (do LAB5):')
print(ragas_df.round(4))
if LANGFUSE_OK:
    trace.score(name='RAGAS_CP_baseline',   value=float(cfg5['cp_baseline']))
    trace.score(name='RAGAS_CP_contextual', value=float(cfg5['cp_contextual']))
    trace.score(name='RAGAS_CR_baseline',   value=float(cfg5['cr_baseline']))
    trace.score(name='RAGAS_CR_contextual', value=float(cfg5['cr_contextual']))
    trace.score(name='RAGAS_delta_CP',
                value=float(cfg5['cp_contextual'] - cfg5['cp_baseline']),
                comment='Δ Context Precision (contextual − baseline)')
    lf.flush()

## 7. Dashboard Visual

In [ ]:
ESTRAT = list(STRATEGIES.keys())
CORES  = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig = plt.figure(figsize=(16, 11))
fig.suptitle('Aula 4 — Benchmark de RAG: Melhoria com Hybrid, Neural Sparse e Contextual Retrieval\n'
             'Corpus TCU 2026 (1.100 acórdãos) — 20 queries — MBA RAG & CAG',
             fontsize=13, fontweight='bold', y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.4)

def get(col, e):
    return agg.loc[e, col]

# MRR@10
ax1 = fig.add_subplot(gs[0, :2])
ax1.bar(ESTRAT, [get('MRR@10', e) for e in ESTRAT], color=CORES, alpha=0.85, edgecolor='black')
ax1.set_title('MRR@10 por Estratégia', fontweight='bold')
ax1.set_ylim(0, 1); ax1.grid(axis='y', alpha=0.3)
for i, e in enumerate(ESTRAT):
    ax1.text(i, get('MRR@10', e) + 0.02, f"{get('MRR@10', e):.3f}", ha='center', fontweight='bold')
ax1.tick_params(axis='x', rotation=20)

# Latência
ax2 = fig.add_subplot(gs[0, 2])
ax2.barh(ESTRAT, [get('latencia_ms', e) for e in ESTRAT], color=CORES, alpha=0.85, edgecolor='black')
ax2.set_title('Latência (ms)', fontweight='bold'); ax2.grid(axis='x', alpha=0.3)
for i, e in enumerate(ESTRAT):
    ax2.text(get('latencia_ms', e) + 1, i, f"{get('latencia_ms', e):.0f}ms", va='center', fontsize=9)

# Δ vs BM25 (MRR/Recall/NDCG)
ax3 = fig.add_subplot(gs[1, :])
x = np.arange(len(ESTRAT)); w = 0.27
for off, col, c in [(-w, '%↑MRR vs BM25', '#4C72B0'),
                     (0,   '%↑Rec vs BM25', '#55A868'),
                     (w,   '%↑NDCG vs BM25', '#DD8452')]:
    vals = [delta.loc[e, col] if e in delta.index else 0 for e in ESTRAT]
    ax3.bar(x + off, vals, w, label=col, color=c, alpha=0.85, edgecolor='black')
ax3.set_xticks(x); ax3.set_xticklabels(ESTRAT, rotation=15)
ax3.set_title('Melhoria do RAG vs BM25 (% por métrica)', fontweight='bold')
ax3.axhline(0, color='black', linewidth=0.6); ax3.grid(axis='y', alpha=0.3)
ax3.legend()

# RAGAS
ax4 = fig.add_subplot(gs[2, :2])
labels  = ragas_df['estrategia']
x = np.arange(len(labels)); w = 0.35
ax4.bar(x - w/2, ragas_df['Context Precision'], w, label='Context Precision',
        color='#4C72B0', alpha=0.85, edgecolor='black')
ax4.bar(x + w/2, ragas_df['Context Recall'],    w, label='Context Recall',
        color='#55A868', alpha=0.85, edgecolor='black')
ax4.set_xticks(x); ax4.set_xticklabels(labels, rotation=10)
ax4.set_ylim(0, 1.05); ax4.grid(axis='y', alpha=0.3)
ax4.set_title('RAGAS: Contextual Retrieval (#T09) — LAB5', fontweight='bold'); ax4.legend()

# Tabela-resumo
ax5 = fig.add_subplot(gs[2, 2]); ax5.axis('off')
tbl = agg.round(3).reset_index()
ax5.table(cellText=tbl.values, colLabels=tbl.columns,
          loc='center', cellLoc='center').auto_set_font_size(False)
ax5.set_title('Resumo (médias)', fontweight='bold')

plt.savefig('dashboard_aula4.png', dpi=150, bbox_inches='tight')
plt.show()
print('dashboard_aula4.png gravado.')

## 8. Exportar CSVs e Persistir

In [ ]:
df.to_csv('benchmark_aula4_detalhado.csv', index=False)
agg_full.to_csv('benchmark_aula4_resumo.csv')
ragas_df.to_csv('benchmark_aula4_ragas.csv', index=False)
print('Exportados: benchmark_aula4_detalhado.csv, benchmark_aula4_resumo.csv, benchmark_aula4_ragas.csv')
print('\nMelhor estratégia (MRR@10):')
best = agg_full['MRR@10'].idxmax()
print(f"  {best} → MRR={agg_full.loc[best, 'MRR@10']:.4f} | "
      f"Recall={agg_full.loc[best, 'Recall@10']:.4f} | NDCG={agg_full.loc[best, 'NDCG@10']:.4f}")

## 9. Conclusão da Aula 4

Nesta aula construímos e medimos:

1. **LAB1**: Índice híbrido + connector Ollama BGE-M3 (1024d) via ML Commons
2. **LAB2**: Search pipelines RRF e Min-Max
3. **LAB3**: Avaliação MRR/Recall/NDCG em 20 queries TCU 2026 com LangFuse
4. **LAB4**: Neural Sparse Search com modelo pré-treinado multilíngue
5. **LAB5**: Contextual Retrieval (#T09) com Groq + Ollama, ganho RAGAS medido
6. **LAB6** (este): consolidação — Hybrid e Contextual superam BM25 nas métricas-chave; Neural Sparse oferece interpretabilidade + boa relação custo/qualidade

**Técnicas dominadas:** #T04 Hybrid Search · #T09 Contextual Retrieval · Neural Sparse (SPLADE)

## Referências (ABNT)

OPENSEARCH PROJECT. **Hybrid Search**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/hybrid-search/>.

OPENSEARCH PROJECT. **Neural sparse search**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/neural-sparse-search/>.

ANTHROPIC. **Introducing Contextual Retrieval**. 2024. <https://www.anthropic.com/news/contextual-retrieval>.

ES, S. et al. **RAGAS: Automated Evaluation of Retrieval Augmented Generation**. arXiv:2309.15217, 2023.

LANGFUSE. **Scores API**. <https://langfuse.com/docs/scores>.